In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, DoubleType, BooleanType, TimestampType

def create_schema():
    return StructType([
        StructField("machine_id", StringType(), True),
        StructField("location_id", StringType(), True),
        StructField("city", StringType(), True),
        StructField("installation_date", DateType(), True),
        StructField("machine_type", StringType(), True),
        StructField("status", StringType(), True),
        StructField("total_brews", IntegerType(), True),
        StructField("daily_avg_brews", DoubleType(), True),
        StructField("last_maintenance_date", DateType(), True),
        StructField("next_maintenance_due", DateType(), True),
        StructField("water_level_liters", DoubleType(), True),
        StructField("bean_level_grams", DoubleType(), True),
        StructField("error_code", StringType(), True),
        StructField("is_operational", BooleanType(), True),
        StructField("revenue_generated", DoubleType(), True),
        StructField("Data_fetch_time", TimestampType(), True)
    ])

In [2]:
def create_dataframe(spark, file_path):
    schema = create_schema()
    return spark.read.csv(file_path, header=True, schema=schema)

In [9]:
def filter_active_maintenance(df):
    return df.filter(
        (F.col("status") == "maintenance") & 
        (F.col("last_maintenance_date") >= "2024-01-10") & 
        (F.col("last_maintenance_date") <= "2024-03-31")
    )

In [10]:
def filter_high_brews(df):
    return df.filter(
        (F.col("total_brews") > 15000) & 
        F.col("status").isNotNull()
    )

In [11]:
def handle_null_maintenance(df):
    return df.withColumn(
        "next_maintenance_due", 
        F.coalesce(F.col("next_maintenance_due"), F.date_add(F.col("last_maintenance_date"), 30))
    )

In [12]:
def add_working_days(df):
    return df.withColumn(
        "working_days", 
        F.datediff(F.current_date(), F.col("last_maintenance_date"))
    )

In [13]:
def add_working_duration(df):
    return df.withColumn(
        "working_duration", 
        F.datediff(F.col("next_maintenance_due"), F.col("last_maintenance_date"))
    )

In [14]:
def add_contract_void(df):
    return df.withColumn(
        "contract_void", 
        F.when(F.col("working_duration") > 40, 1).otherwise(0)
    )

In [15]:
def get_coffee_machine_types(df):
    return df.filter(F.col("city").isin("Pune", "Mumbai", "Delhi")) \
             .select("machine_type") \
             .distinct() \
             .rdd.flatMap(lambda x: x) \
             .collect()

In [16]:
def machine_efficiency(df):
    total_val = df.select(F.sum("total_brews")).first()[0]
    if total_val is None:
        total_val = 0
    return df.withColumn(
        "opti", 
        F.when(F.col("daily_avg_brews") == 0.0, 0.0).otherwise(total_val / F.col("daily_avg_brews"))
    )

In [17]:
def get_most_popular_machine(df):
    return df.groupBy("machine_type") \
             .count() \
             .orderBy(F.col("count").desc()) \
             .select("machine_type") \
             .first()[0]

In [18]:
def shortlist_cities(df):
    return df.filter(F.col("status") == "active") \
             .groupBy("city") \
             .agg(F.count("*").alias("mas")) \
             .orderBy(F.col("mas").desc(), F.col("city").asc())

In [19]:
def filter_specific_status(df):
    status = ["active", "inactive"]
    return df.filter(F.col("status").isin(status))

In [20]:
def recently_maintained_active(df):
    return df.filter(F.col("status") == "active") \
             .orderBy(F.col("last_maintenance_date").desc()) \
             .limit(1)

In [21]:
def clean_df(df):
    return df.dropna(subset=["total_brews", "next_maintenance_due"])

In [22]:
def best_contribution(df):
    total_all = df.select(F.sum("total_brews")).first()[0]
    if total_all is None or total_all == 0:
        total_all = 1
        
    return df.groupBy("city") \
             .agg(F.sum("total_brews").alias("total_by_city")) \
             .withColumn("contribution", F.col("total_by_city") / total_all) \
             .orderBy("city") \
             .select("city") \
             .rdd.flatMap(lambda x: x) \
             .collect()

In [23]:
def peak_serves_per_city(df):
    return df.groupBy("city") \
             .agg(F.max("total_brews").alias("peak_serves")) \
             .orderBy(F.col("peak_serves").desc()) \
             .limit(3)

In [24]:
def odd_maintenance(df):
    latest_date = df.select(F.max("installation_date")).first()[0]
    return df.withColumn("age", F.datediff(F.lit(latest_date), F.col("installation_date"))) \
             .filter(F.col("age") <= 20) \
             .drop("age")

In [25]:
def data_age_days(df):
    return df.withColumn(
        "data_age", 
        F.round((F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.col("Data_fetch_time"))) / (24 * 3600)).cast(IntegerType())
    )

In [26]:
def data_age_hours(df):
    return df.withColumn(
        "data_age", 
        F.round((F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.col("Data_fetch_time"))) / 3600).cast(IntegerType())
    )